In [4]:
# ============================================================
# CELL 1 — IMPORTS AND LOAD RAW UK-DALE DATASET
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from nilmtk import DataSet

# ------------------------------------------------------------
# Dataset path
# ------------------------------------------------------------
DATASET_PATH = "../data/processed/ukdale.h5"

# ------------------------------------------------------------
# Load raw dataset
# ------------------------------------------------------------
ds = DataSet(DATASET_PATH)

print("UK-DALE dataset loaded successfully.")
print("Available buildings:", list(ds.buildings.keys()))

UK-DALE dataset loaded successfully.
Available buildings: [1, 2, 3, 4, 5]


In [5]:
# ============================================================
# CELL 2 — DEFINE FUNDAMENTAL CLEANING CONFIGURATION
# ============================================================

# Houses selected for the project
HOUSE_IDS = [1, 2, 5]

# Working sampling interval
SAMPLE_PERIOD = 6  # seconds

# Power measurement configuration
PHYSICAL_QUANTITY = "power"
AC_TYPE = "active"

# ------------------------------------------------------------
# Candidate appliance meters for each house
# ------------------------------------------------------------
APPLIANCE_METERS = {
    1: {
        "dishwasher": 6,
        "television": 7,
        "fridge_freezer": 12,
        "microwave": 13,
        "oven": 42
    },

    2: {
        "kettle": 8,
        "rice_cooker": 9,
        "washing_machine": 12,
        "dishwasher": 13,
        "fridge": 14
    },

    5: {
        "washer_dryer": 24,
        "fridge_freezer": 19,
        "electric_oven": 20
    }
}

# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------
print("Fundamental Cleaning Configuration")
print("=" * 55)

print(f"Houses selected: {HOUSE_IDS}")
print(f"Sampling period: {SAMPLE_PERIOD} seconds")
print(f"Physical quantity: {PHYSICAL_QUANTITY}")
print(f"AC type: {AC_TYPE}")

print("\nCandidate appliances:")

for house_id in HOUSE_IDS:
    print(f"\nHouse {house_id}:")
    
    for appliance, meter_id in APPLIANCE_METERS[house_id].items():
        print(f"  {appliance}: meter {meter_id}")

Fundamental Cleaning Configuration
Houses selected: [1, 2, 5]
Sampling period: 6 seconds
Physical quantity: power
AC type: active

Candidate appliances:

House 1:
  dishwasher: meter 6
  television: meter 7
  fridge_freezer: meter 12
  microwave: meter 13
  oven: meter 42

House 2:
  kettle: meter 8
  rice_cooker: meter 9
  washing_machine: meter 12
  dishwasher: meter 13
  fridge: meter 14

House 5:
  washer_dryer: meter 24
  fridge_freezer: meter 19
  electric_oven: meter 20


In [6]:
# ============================================================
# CELL 3 — LOAD RAW SELECTED SIGNALS
# ============================================================

raw_data = {}

for house_id in HOUSE_IDS:
    print(f"\nLoading House {house_id}...")
    
    house = ds.buildings[house_id]
    
    # --------------------------------------------------------
    # Load active mains
    # --------------------------------------------------------
    mains_df = next(
        house.elec.mains().load(
            physical_quantity=PHYSICAL_QUANTITY,
            ac_type=AC_TYPE,
            sample_period=SAMPLE_PERIOD
        )
    )
    
    mains_series = mains_df.iloc[:, 0]
    
    # Store mains
    raw_data[house_id] = {
        "mains": mains_series
    }
    
    print(f"  Mains loaded: {len(mains_series):,} samples")
    
    # --------------------------------------------------------
    # Load candidate appliances
    # --------------------------------------------------------
    for appliance, meter_id in APPLIANCE_METERS[house_id].items():
        
        appliance_df = next(
            house.elec[meter_id].load(
                physical_quantity=PHYSICAL_QUANTITY,
                ac_type=AC_TYPE,
                sample_period=SAMPLE_PERIOD
            )
        )
        
        appliance_series = appliance_df.iloc[:, 0]
        
        raw_data[house_id][appliance] = appliance_series
        
        print(
            f"  {appliance:<18} "
            f"meter {meter_id:<3} "
            f"{len(appliance_series):,} samples"
        )

print("\nRaw signal loading completed.")


Loading House 1...
  Mains loaded: 21,613,433 samples
  dishwasher         meter 6   23,454,673 samples
  television         meter 7   23,454,674 samples
  fridge_freezer     meter 12  22,950,714 samples
  microwave          meter 13  22,950,716 samples
  oven               meter 42  21,655,948 samples

Loading House 2...
  Mains loaded: 2,539,509 samples
  kettle             meter 8   3,377,557 samples
  rice_cooker        meter 9   2,539,179 samples
  washing_machine    meter 12  2,049,468 samples
  dishwasher         meter 13  2,049,468 samples
  fridge             meter 14  2,049,469 samples

Loading House 5...
  Mains loaded: 1,975,318 samples
  washer_dryer       meter 24  1,973,736 samples
  fridge_freezer     meter 19  1,973,743 samples
  electric_oven      meter 20  1,973,650 samples

Raw signal loading completed.


In [7]:
# ============================================================
# CELL 4 — RAW DATA QUALITY SUMMARY
# ============================================================

def signal_quality_summary(series):
    """
    Return basic quality statistics for a time-series signal.
    No data is modified.
    """
    
    index = series.index
    values = series.to_numpy()
    
    # Sampling intervals
    intervals = index.to_series().diff().dropna()
    
    interval_counts = intervals.value_counts()
    
    if len(interval_counts) > 0:
        most_common_interval = interval_counts.index[0]
        irregular_intervals = (intervals != most_common_interval).sum()
    else:
        most_common_interval = None
        irregular_intervals = 0
    
    return {
        "samples": len(series),
        "start": index.min(),
        "end": index.max(),
        "NaN": series.isna().sum(),
        "Inf": np.isinf(values).sum(),
        "negative": (series < 0).sum(),
        "zero": (series == 0).sum(),
        "min_W": series.min(),
        "max_W": series.max(),
        "common_interval": most_common_interval,
        "irregular_intervals": irregular_intervals
    }


# ------------------------------------------------------------
# Generate summary for every signal
# ------------------------------------------------------------

quality_rows = []

for house_id in HOUSE_IDS:
    
    for signal_name, series in raw_data[house_id].items():
        
        stats = signal_quality_summary(series)
        
        stats["house"] = house_id
        stats["signal"] = signal_name
        
        quality_rows.append(stats)


quality_df = pd.DataFrame(quality_rows)

# Reorder columns
quality_df = quality_df[
    [
        "house",
        "signal",
        "samples",
        "start",
        "end",
        "NaN",
        "Inf",
        "negative",
        "zero",
        "min_W",
        "max_W",
        "common_interval",
        "irregular_intervals"
    ]
]

# Display
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

display(quality_df)

,house,signal,samples,start,end,NaN,Inf,negative,zero,min_W,max_W,common_interval,irregular_intervals
0,1,mains,21613433,2013-03-17 19:12:42+00:00,2017-04-26 18:35:54+01:00,252881,0,0,0,36.141666,7997.665527,0 days 00:00:06,0
1,1,dishwasher,23454673,2012-11-09 22:28:18+00:00,2017-04-26 18:35:30+01:00,702237,0,0,3986287,0.000000,3973.000000,0 days 00:00:06,0
2,1,television,23454674,2012-11-09 22:28:18+00:00,2017-04-26 18:35:36+01:00,735106,0,0,1072735,0.000000,3109.000000,0 days 00:00:06,0
3,1,fridge_freezer,22950714,2012-12-14 22:21:30+00:00,2017-04-26 18:32:48+01:00,502629,0,0,12890750,0.000000,3323.000000,0 days 00:00:06,0
4,1,microwave,22950716,2012-12-14 22:21:30+00:00,2017-04-26 18:33:00+01:00,498874,0,0,140735,0.000000,3267.000000,0 days 00:00:06,0
5,1,oven,21655948,2013-03-14 20:20:00+00:00,2017-04-26 18:34:42+01:00,506383,0,0,53,0.000000,1070.000000,0 days 00:00:06,0
6,2,mains,2539509,2013-04-16 21:45:12+01:00,2013-10-10 06:16:00+01:00,512781,0,0,6521,0.000000,6690.669922,0 days 00:00:06,0
7,2,kettle,3377557,2013-02-17 16:00:18+00:00,2013-10-10 06:15:54+01:00,1231651,0,0,153864,0.000000,3998.000000,0 days 00:00:06,0
8,2,rice_cooker,2539179,2013-04-16 22:18:06+01:00,2013-10-10 06:15:54+01:00,395805,0,0,10617,0.000000,1017.000000,0 days 00:00:06,0
9,2,washing_machine,2049468,2013-05-20 22:28:36+01:00,2013-10-10 06:15:18+01:00,321742,0,0,890,0.000000,2974.000000,0 days 00:00:06,0


In [8]:
# ============================================================
# CELL 5 — CHECK COMMON TIME OVERLAP
# ============================================================

overlap_rows = []

for house_id in HOUSE_IDS:

    mains = raw_data[house_id]["mains"]

    print(f"\nHouse {house_id}")
    print("=" * 60)

    for appliance in APPLIANCE_METERS[house_id]:

        appliance_series = raw_data[house_id][appliance]

        # ----------------------------------------------------
        # Find overlapping time period
        # ----------------------------------------------------
        overlap_start = max(
            mains.index.min(),
            appliance_series.index.min()
        )

        overlap_end = min(
            mains.index.max(),
            appliance_series.index.max()
        )

        # ----------------------------------------------------
        # Find timestamps present in both signals
        # ----------------------------------------------------
        common_index = mains.index.intersection(
            appliance_series.index
        )

        # Keep only timestamps inside the overlap
        common_index = common_index[
            (common_index >= overlap_start) &
            (common_index <= overlap_end)
        ]

        row = {
            "house": house_id,
            "appliance": appliance,
            "overlap_start": overlap_start,
            "overlap_end": overlap_end,
            "common_samples": len(common_index),
            "mains_samples": len(mains.loc[common_index]),
            "appliance_samples": len(
                appliance_series.loc[common_index]
            )
        }

        overlap_rows.append(row)

        print(
            f"{appliance:<18} | "
            f"{overlap_start} → {overlap_end} | "
            f"{len(common_index):,} common samples"
        )


overlap_df = pd.DataFrame(overlap_rows)

print("\n")
print("OVERLAP SUMMARY")
print("=" * 80)

display(overlap_df)


House 1
dishwasher         | 2013-03-17 19:12:42+00:00 → 2017-04-26 18:35:30+01:00 | 21,613,429 common samples
television         | 2013-03-17 19:12:42+00:00 → 2017-04-26 18:35:36+01:00 | 21,613,430 common samples
fridge_freezer     | 2013-03-17 19:12:42+00:00 → 2017-04-26 18:32:48+01:00 | 21,613,402 common samples
microwave          | 2013-03-17 19:12:42+00:00 → 2017-04-26 18:33:00+01:00 | 21,613,404 common samples
oven               | 2013-03-17 19:12:42+00:00 → 2017-04-26 18:34:42+01:00 | 21,613,421 common samples

House 2
kettle             | 2013-04-16 21:45:12+01:00 → 2013-10-10 06:15:54+01:00 | 2,539,508 common samples
rice_cooker        | 2013-04-16 22:18:06+01:00 → 2013-10-10 06:15:54+01:00 | 2,539,179 common samples
washing_machine    | 2013-05-20 22:28:36+01:00 → 2013-10-10 06:15:18+01:00 | 2,049,468 common samples
dishwasher         | 2013-05-20 22:28:36+01:00 → 2013-10-10 06:15:18+01:00 | 2,049,468 common samples
fridge             | 2013-05-20 22:28:36+01:00 → 2013-10-10

,house,appliance,overlap_start,overlap_end,common_samples,mains_samples,appliance_samples
0,1,dishwasher,2013-03-17 19:12:42+00:00,2017-04-26 18:35:30+01:00,21613429,21613429,21613429
1,1,television,2013-03-17 19:12:42+00:00,2017-04-26 18:35:36+01:00,21613430,21613430,21613430
2,1,fridge_freezer,2013-03-17 19:12:42+00:00,2017-04-26 18:32:48+01:00,21613402,21613402,21613402
3,1,microwave,2013-03-17 19:12:42+00:00,2017-04-26 18:33:00+01:00,21613404,21613404,21613404
4,1,oven,2013-03-17 19:12:42+00:00,2017-04-26 18:34:42+01:00,21613421,21613421,21613421
5,2,kettle,2013-04-16 21:45:12+01:00,2013-10-10 06:15:54+01:00,2539508,2539508,2539508
6,2,rice_cooker,2013-04-16 22:18:06+01:00,2013-10-10 06:15:54+01:00,2539179,2539179,2539179
7,2,washing_machine,2013-05-20 22:28:36+01:00,2013-10-10 06:15:18+01:00,2049468,2049468,2049468
8,2,dishwasher,2013-05-20 22:28:36+01:00,2013-10-10 06:15:18+01:00,2049468,2049468,2049468
9,2,fridge,2013-05-20 22:28:36+01:00,2013-10-10 06:15:24+01:00,2049469,2049469,2049469


In [9]:
# ============================================================
# CELL 6 — ANALYZE MISSING-VALUE ALIGNMENT
# ============================================================

missing_rows = []

for house_id in HOUSE_IDS:

    mains = raw_data[house_id]["mains"]

    print(f"\nHouse {house_id}")
    print("=" * 70)

    for appliance in APPLIANCE_METERS[house_id]:

        appliance_series = raw_data[house_id][appliance]

        # ----------------------------------------------------
        # Common timestamps
        # ----------------------------------------------------
        common_index = mains.index.intersection(
            appliance_series.index
        )

        mains_common = mains.loc[common_index]
        appliance_common = appliance_series.loc[common_index]

        # ----------------------------------------------------
        # Missing values
        # ----------------------------------------------------
        mains_nan = mains_common.isna()
        appliance_nan = appliance_common.isna()

        both_nan = mains_nan & appliance_nan
        mains_only_nan = mains_nan & ~appliance_nan
        appliance_only_nan = ~mains_nan & appliance_nan
        neither_nan = ~mains_nan & ~appliance_nan

        row = {
            "house": house_id,
            "appliance": appliance,
            "common_samples": len(common_index),
            "mains_nan": mains_nan.sum(),
            "appliance_nan": appliance_nan.sum(),
            "both_nan": both_nan.sum(),
            "mains_only_nan": mains_only_nan.sum(),
            "appliance_only_nan": appliance_only_nan.sum(),
            "both_valid": neither_nan.sum()
        }

        missing_rows.append(row)

        print(
            f"{appliance:<18} | "
            f"Mains NaN: {mains_nan.sum():>9,} | "
            f"Appliance NaN: {appliance_nan.sum():>9,} | "
            f"Both NaN: {both_nan.sum():>9,} | "
            f"Both valid: {neither_nan.sum():>9,}"
        )


missing_df = pd.DataFrame(missing_rows)

print("\n")
print("MISSING-VALUE ALIGNMENT SUMMARY")
print("=" * 90)

display(missing_df)


House 1
dishwasher         | Mains NaN:   252,881 | Appliance NaN:   158,109 | Both NaN:    15,131 | Both valid: 21,217,570
television         | Mains NaN:   252,881 | Appliance NaN:   159,511 | Both NaN:    15,115 | Both valid: 21,216,153
fridge_freezer     | Mains NaN:   252,881 | Appliance NaN:   169,544 | Both NaN:    15,328 | Both valid: 21,206,305
microwave          | Mains NaN:   252,881 | Appliance NaN:   166,126 | Both NaN:    15,155 | Both valid: 21,209,552
oven               | Mains NaN:   252,881 | Appliance NaN:   506,009 | Both NaN:    15,139 | Both valid: 20,869,670

House 2
kettle             | Mains NaN:   512,781 | Appliance NaN:   393,630 | Both NaN:   298,817 | Both valid: 1,931,914
rice_cooker        | Mains NaN:   512,537 | Appliance NaN:   395,805 | Both NaN:   299,380 | Both valid: 1,930,217
washing_machine    | Mains NaN:   509,385 | Appliance NaN:   321,742 | Both NaN:   295,497 | Both valid: 1,513,838
dishwasher         | Mains NaN:   509,385 | Appliance NaN

,house,appliance,common_samples,mains_nan,appliance_nan,both_nan,mains_only_nan,appliance_only_nan,both_valid
0,1,dishwasher,21613429,252881,158109,15131,237750,142978,21217570
1,1,television,21613430,252881,159511,15115,237766,144396,21216153
2,1,fridge_freezer,21613402,252881,169544,15328,237553,154216,21206305
3,1,microwave,21613404,252881,166126,15155,237726,150971,21209552
4,1,oven,21613421,252881,506009,15139,237742,490870,20869670
5,2,kettle,2539508,512781,393630,298817,213964,94813,1931914
6,2,rice_cooker,2539179,512537,395805,299380,213157,96425,1930217
7,2,washing_machine,2049468,509385,321742,295497,213888,26245,1513838
8,2,dishwasher,2049468,509385,321653,295496,213889,26157,1513926
9,2,fridge,2049469,509385,321660,295497,213888,26163,1513921


In [10]:
# ============================================================
# CELL 7 — ANALYZE CONSECUTIVE NaN GAPS
# ============================================================

def get_nan_gap_lengths(series):
    """
    Find lengths of consecutive NaN runs.
    Returns gap lengths in number of samples.
    """
    
    is_nan = series.isna()
    
    # Identify starts/ends of NaN runs
    groups = (is_nan != is_nan.shift()).cumsum()
    
    gap_lengths = (
        is_nan
        .groupby(groups)
        .sum()
    )
    
    # Keep only groups that are actually NaN
    gap_lengths = gap_lengths[gap_lengths > 0]
    
    return gap_lengths.astype(int)


gap_rows = []

for house_id in HOUSE_IDS:

    print(f"\nHouse {house_id}")
    print("=" * 80)

    for signal_name, series in raw_data[house_id].items():

        gap_lengths = get_nan_gap_lengths(series)

        if len(gap_lengths) == 0:
            print(f"{signal_name:<20} No NaN gaps")
            continue

        # Convert samples → seconds
        gap_seconds = gap_lengths * SAMPLE_PERIOD

        row = {
            "house": house_id,
            "signal": signal_name,
            "number_of_nan_gaps": len(gap_lengths),
            "shortest_gap_samples": gap_lengths.min(),
            "median_gap_samples": gap_lengths.median(),
            "longest_gap_samples": gap_lengths.max(),
            "shortest_gap_seconds": gap_seconds.min(),
            "median_gap_seconds": gap_seconds.median(),
            "longest_gap_seconds": gap_seconds.max()
        }

        gap_rows.append(row)

        print(
            f"{signal_name:<20} "
            f"gaps: {len(gap_lengths):>6,} | "
            f"median: {gap_seconds.median():>8.0f} sec | "
            f"longest: {gap_seconds.max():>10.0f} sec"
        )


gap_df = pd.DataFrame(gap_rows)

print("\n")
print("NaN GAP SUMMARY")
print("=" * 100)

display(gap_df)


House 1
mains                gaps:     41 | median:       48 sec | longest:     884940 sec
dishwasher           gaps:    187 | median:      366 sec | longest:     788406 sec
television           gaps:    431 | median:      222 sec | longest:     788742 sec
fridge_freezer       gaps:    736 | median:       78 sec | longest:     626652 sec
microwave            gaps:    534 | median:       87 sec | longest:     626646 sec
oven                 gaps:    466 | median:       54 sec | longest:    1121334 sec

House 2
mains                gaps:     12 | median:      288 sec | longest:    2968476 sec
kettle               gaps:     10 | median:   122085 sec | longest:    5030100 sec
rice_cooker          gaps:     88 | median:      138 sec | longest:    1464144 sec
washing_machine      gaps:     11 | median:      372 sec | longest:    1464144 sec
dishwasher           gaps:      6 | median:   122085 sec | longest:    1464144 sec
fridge               gaps:      7 | median:    87252 sec | longest:  

,house,signal,number_of_nan_gaps,shortest_gap_samples,median_gap_samples,longest_gap_samples,shortest_gap_seconds,median_gap_seconds,longest_gap_seconds
0,1,mains,41,1,8.0,147490,6,48.0,884940
1,1,dishwasher,187,1,61.0,131401,6,366.0,788406
2,1,television,431,1,37.0,131457,6,222.0,788742
3,1,fridge_freezer,736,1,13.0,104442,6,78.0,626652
4,1,microwave,534,1,14.5,104441,6,87.0,626646
5,1,oven,466,1,9.0,186889,6,54.0,1121334
6,2,mains,12,3,48.0,494746,18,288.0,2968476
7,2,kettle,10,62,20347.5,838350,372,122085.0,5030100
8,2,rice_cooker,88,1,23.0,244024,6,138.0,1464144
9,2,washing_machine,11,4,62.0,244024,24,372.0,1464144


In [11]:
# ============================================================
# CELL 8 — CLASSIFY NaN GAPS BY DURATION
# ============================================================

# Conservative threshold for a "short" gap
SHORT_GAP_SECONDS = 300  # 5 minutes

classification_rows = []

for house_id in HOUSE_IDS:

    print(f"\nHouse {house_id}")
    print("=" * 90)

    for signal_name, series in raw_data[house_id].items():

        gap_lengths = get_nan_gap_lengths(series)

        if len(gap_lengths) == 0:
            continue

        gap_seconds = gap_lengths * SAMPLE_PERIOD

        short_gaps = gap_seconds[gap_seconds <= SHORT_GAP_SECONDS]
        long_gaps = gap_seconds[gap_seconds > SHORT_GAP_SECONDS]

        short_samples = gap_lengths[gap_seconds <= SHORT_GAP_SECONDS].sum()
        long_samples = gap_lengths[gap_seconds > SHORT_GAP_SECONDS].sum()

        total_nan = series.isna().sum()

        row = {
            "house": house_id,
            "signal": signal_name,
            "total_nan_samples": total_nan,

            "short_gap_count": len(short_gaps),
            "short_gap_samples": int(short_samples),
            "short_gap_seconds": int(short_samples * SAMPLE_PERIOD),

            "long_gap_count": len(long_gaps),
            "long_gap_samples": int(long_samples),
            "long_gap_seconds": int(long_samples * SAMPLE_PERIOD),

            "short_gap_%_of_nan": (
                short_samples / total_nan * 100
                if total_nan > 0 else 0
            ),

            "long_gap_%_of_nan": (
                long_samples / total_nan * 100
                if total_nan > 0 else 0
            )
        }

        classification_rows.append(row)

        print(
            f"{signal_name:<20} | "
            f"NaNs: {total_nan:>10,} | "
            f"short gaps: {len(short_gaps):>4} "
            f"({short_samples:>10,} samples) | "
            f"long gaps: {len(long_gaps):>4} "
            f"({long_samples:>10,} samples)"
        )


gap_classification_df = pd.DataFrame(classification_rows)

print("\n")
print("NaN GAP CLASSIFICATION SUMMARY")
print("=" * 110)

display(gap_classification_df)


House 1
mains                | NaNs:    252,881 | short gaps:   28 (       184 samples) | long gaps:   13 (   252,697 samples)
dishwasher           | NaNs:    702,237 | short gaps:   84 (     1,366 samples) | long gaps:  103 (   700,871 samples)
television           | NaNs:    735,106 | short gaps:  261 (     4,710 samples) | long gaps:  170 (   730,396 samples)
fridge_freezer       | NaNs:    502,629 | short gaps:  596 (     7,970 samples) | long gaps:  140 (   494,659 samples)
microwave            | NaNs:    498,874 | short gaps:  407 (     5,503 samples) | long gaps:  127 (   493,371 samples)
oven                 | NaNs:    506,383 | short gaps:  404 (     5,043 samples) | long gaps:   62 (   501,340 samples)

House 2
mains                | NaNs:    512,781 | short gaps:    6 (        53 samples) | long gaps:    6 (   512,728 samples)
kettle               | NaNs:  1,231,651 | short gaps:    0 (         0 samples) | long gaps:   10 ( 1,231,651 samples)
rice_cooker          | NaNs:  

,house,signal,total_nan_samples,short_gap_count,short_gap_samples,short_gap_seconds,long_gap_count,long_gap_samples,long_gap_seconds,short_gap_%_of_nan,long_gap_%_of_nan
0,1,mains,252881,28,184,1104,13,252697,1516182,0.072761,99.927239
1,1,dishwasher,702237,84,1366,8196,103,700871,4205226,0.194521,99.805479
2,1,television,735106,261,4710,28260,170,730396,4382376,0.640724,99.359276
3,1,fridge_freezer,502629,596,7970,47820,140,494659,2967954,1.585663,98.414337
4,1,microwave,498874,407,5503,33018,127,493371,2960226,1.103084,98.896916
5,1,oven,506383,404,5043,30258,62,501340,3008040,0.995887,99.004113
6,2,mains,512781,6,53,318,6,512728,3076368,0.010336,99.989664
7,2,kettle,1231651,0,0,0,10,1231651,7389906,0.000000,100.000000
8,2,rice_cooker,395805,62,965,5790,26,394840,2369040,0.243807,99.756193
9,2,washing_machine,321742,4,32,192,7,321710,1930260,0.009946,99.990054


In [12]:
# ============================================================
# CELL 9 — INSPECT LOCATIONS OF SHORT NaN GAPS
# ============================================================

SHORT_GAP_SECONDS = 300  # 5 minutes
SHORT_GAP_SAMPLES = SHORT_GAP_SECONDS // SAMPLE_PERIOD


def get_nan_gap_info(series):
    """
    Return information about every consecutive NaN gap.
    """
    
    is_nan = series.isna()
    
    groups = (is_nan != is_nan.shift()).cumsum()
    
    gap_info = []
    
    for group_id, group in series.groupby(groups):
        
        if not group.isna().all():
            continue
        
        gap_length = len(group)
        
        gap_info.append({
            "start": group.index.min(),
            "end": group.index.max(),
            "samples": gap_length,
            "seconds": gap_length * SAMPLE_PERIOD
        })
    
    return pd.DataFrame(gap_info)


short_gap_rows = []

for house_id in HOUSE_IDS:

    print(f"\nHouse {house_id}")
    print("=" * 90)

    for signal_name, series in raw_data[house_id].items():

        gap_info = get_nan_gap_info(series)

        if gap_info.empty:
            continue

        short_gaps = gap_info[
            gap_info["seconds"] <= SHORT_GAP_SECONDS
        ].copy()

        if short_gaps.empty:
            print(f"{signal_name:<20} No short gaps")
            continue

        # ----------------------------------------------------
        # Check whether each short gap is surrounded by
        # valid data on both sides.
        # ----------------------------------------------------
        isolated_count = 0
        edge_count = 0

        for _, gap in short_gaps.iterrows():

            gap_start = gap["start"]
            gap_end = gap["end"]

            try:
                position_start = series.index.get_loc(gap_start)
                position_end = series.index.get_loc(gap_end)

                has_previous = position_start > 0
                has_next = position_end < len(series) - 1

                previous_valid = (
                    has_previous and
                    not pd.isna(series.iloc[position_start - 1])
                )

                next_valid = (
                    has_next and
                    not pd.isna(series.iloc[position_end + 1])
                )

                if previous_valid and next_valid:
                    isolated_count += 1
                else:
                    edge_count += 1

            except Exception:
                edge_count += 1

        short_gap_rows.append({
            "house": house_id,
            "signal": signal_name,
            "short_gap_count": len(short_gaps),
            "isolated_short_gaps": isolated_count,
            "edge_or_unbounded_gaps": edge_count,
            "short_gap_samples": int(short_gaps["samples"].sum()),
            "short_gap_seconds": int(
                short_gaps["seconds"].sum()
            )
        })

        print(
            f"{signal_name:<20} | "
            f"short gaps: {len(short_gaps):>4} | "
            f"isolated: {isolated_count:>4} | "
            f"edge/unbounded: {edge_count:>4}"
        )


short_gap_df = pd.DataFrame(short_gap_rows)

print("\n")
print("SHORT GAP LOCATION SUMMARY")
print("=" * 100)

display(short_gap_df)


House 1
mains                | short gaps:   28 | isolated:   28 | edge/unbounded:    0
dishwasher           | short gaps:   84 | isolated:   84 | edge/unbounded:    0
television           | short gaps:  261 | isolated:  261 | edge/unbounded:    0
fridge_freezer       | short gaps:  596 | isolated:  596 | edge/unbounded:    0
microwave            | short gaps:  407 | isolated:  407 | edge/unbounded:    0
oven                 | short gaps:  404 | isolated:  404 | edge/unbounded:    0

House 2
mains                | short gaps:    6 | isolated:    6 | edge/unbounded:    0
kettle               No short gaps
rice_cooker          | short gaps:   62 | isolated:   62 | edge/unbounded:    0
washing_machine      | short gaps:    4 | isolated:    4 | edge/unbounded:    0
dishwasher           No short gaps
fridge               | short gaps:    1 | isolated:    1 | edge/unbounded:    0

House 5
mains                No short gaps
washer_dryer         No short gaps
fridge_freezer       No short gap

,house,signal,short_gap_count,isolated_short_gaps,edge_or_unbounded_gaps,short_gap_samples,short_gap_seconds
0,1,mains,28,28,0,184,1104
1,1,dishwasher,84,84,0,1366,8196
2,1,television,261,261,0,4710,28260
3,1,fridge_freezer,596,596,0,7970,47820
4,1,microwave,407,407,0,5503,33018
5,1,oven,404,404,0,5043,30258
6,2,mains,6,6,0,53,318
7,2,rice_cooker,62,62,0,965,5790
8,2,washing_machine,4,4,0,32,192
9,2,fridge,1,1,0,6,36


In [13]:
# ============================================================
# CELL 10 — FUNDAMENTAL DATA CLEANING
# ============================================================

MAX_INTERPOLATION_SECONDS = 300  # 5 minutes
MAX_INTERPOLATION_SAMPLES = (
    MAX_INTERPOLATION_SECONDS // SAMPLE_PERIOD
)

print("Starting fundamental data cleaning...")
print("=" * 70)

cleaned_data = {}


def interpolate_only_short_gaps(series, max_gap_samples):
    """
    Linearly interpolate only complete NaN gaps whose length
    is <= max_gap_samples.

    Longer NaN gaps are left untouched.
    """

    cleaned = series.copy()

    is_nan = cleaned.isna()

    # Identify consecutive runs of NaNs
    groups = (is_nan != is_nan.shift()).cumsum()

    for group_id, group in cleaned.groupby(groups):

        # Skip non-NaN groups
        if not group.isna().all():
            continue

        gap_length = len(group)

        # Only interpolate short gaps
        if gap_length <= max_gap_samples:

            start_position = cleaned.index.get_loc(
                group.index[0]
            )

            end_position = cleaned.index.get_loc(
                group.index[-1]
            )

            # Safety check: gap must have valid values
            # immediately before and after it.
            if (
                start_position > 0
                and end_position < len(cleaned) - 1
                and not pd.isna(cleaned.iloc[start_position - 1])
                and not pd.isna(cleaned.iloc[end_position + 1])
            ):

                # Interpolate this specific gap
                previous_value = cleaned.iloc[
                    start_position - 1
                ]

                next_value = cleaned.iloc[
                    end_position + 1
                ]

                gap_positions = range(
                    start_position,
                    end_position + 1
                )

                number_of_missing = gap_length

                for i, position in enumerate(gap_positions, start=1):

                    fraction = i / (number_of_missing + 1)

                    cleaned.iloc[position] = (
                        previous_value
                        + fraction * (
                            next_value - previous_value
                        )
                    )

    return cleaned


for house_id in HOUSE_IDS:

    print(f"\nCleaning House {house_id}")
    print("-" * 70)

    cleaned_data[house_id] = {}

    for signal_name, series in raw_data[house_id].items():

        # ----------------------------------------------------
        # Keep raw_data untouched
        # ----------------------------------------------------
        cleaned_series = interpolate_only_short_gaps(
            series,
            MAX_INTERPOLATION_SAMPLES
        )

        cleaned_data[house_id][signal_name] = cleaned_series

        # ----------------------------------------------------
        # Cleaning statistics
        # ----------------------------------------------------
        original_nan = series.isna().sum()
        remaining_nan = cleaned_series.isna().sum()
        interpolated = original_nan - remaining_nan

        print(
            f"{signal_name:<20} | "
            f"original NaN: {original_nan:>10,} | "
            f"interpolated: {interpolated:>10,} | "
            f"remaining NaN: {remaining_nan:>10,}"
        )

print("\n")
print("Fundamental cleaning completed.")

Starting fundamental data cleaning...

Cleaning House 1
----------------------------------------------------------------------
mains                | original NaN:    252,881 | interpolated:        184 | remaining NaN:    252,697
dishwasher           | original NaN:    702,237 | interpolated:      1,366 | remaining NaN:    700,871
television           | original NaN:    735,106 | interpolated:      4,710 | remaining NaN:    730,396
fridge_freezer       | original NaN:    502,629 | interpolated:      7,970 | remaining NaN:    494,659
microwave            | original NaN:    498,874 | interpolated:      5,503 | remaining NaN:    493,371
oven                 | original NaN:    506,383 | interpolated:      5,043 | remaining NaN:    501,340

Cleaning House 2
----------------------------------------------------------------------
mains                | original NaN:    512,781 | interpolated:         53 | remaining NaN:    512,728
kettle               | original NaN:  1,231,651 | interpolated:

In [14]:
# ============================================================
# CELL 11 — VERIFY FUNDAMENTAL CLEANING
# ============================================================

verification_rows = []

for house_id in HOUSE_IDS:

    for signal_name, series in cleaned_data[house_id].items():

        # ----------------------------------------------------
        # Remaining NaN gaps
        # ----------------------------------------------------
        gap_lengths = get_nan_gap_lengths(series)

        if len(gap_lengths) > 0:
            gap_seconds = gap_lengths * SAMPLE_PERIOD

            remaining_short_gaps = (
                gap_seconds <= MAX_INTERPOLATION_SECONDS
            ).sum()

            longest_gap_seconds = gap_seconds.max()

        else:
            remaining_short_gaps = 0
            longest_gap_seconds = 0

        # ----------------------------------------------------
        # Basic value checks
        # ----------------------------------------------------
        negative_values = (series < 0).sum()
        infinite_values = np.isinf(series.to_numpy()).sum()
        zero_values = (series == 0).sum()

        verification_rows.append({
            "house": house_id,
            "signal": signal_name,
            "remaining_NaN": series.isna().sum(),
            "remaining_short_gaps": remaining_short_gaps,
            "longest_remaining_gap_sec": longest_gap_seconds,
            "negative_values": negative_values,
            "infinite_values": infinite_values,
            "zero_values": zero_values
        })


verification_df = pd.DataFrame(verification_rows)

print("FUNDAMENTAL CLEANING VERIFICATION")
print("=" * 100)

display(verification_df)

FUNDAMENTAL CLEANING VERIFICATION


,house,signal,remaining_NaN,remaining_short_gaps,longest_remaining_gap_sec,negative_values,infinite_values,zero_values
0,1,mains,252697,0,884940,0,0,0
1,1,dishwasher,700871,0,788406,0,0,3986362
2,1,television,730396,0,788742,0,0,1072821
3,1,fridge_freezer,494659,0,626652,0,0,12894516
4,1,microwave,493371,0,626646,0,0,140735
5,1,oven,501340,0,1121334,0,0,53
6,2,mains,512728,0,2968476,0,0,6521
7,2,kettle,1231651,0,5030100,0,0,153864
8,2,rice_cooker,394840,0,1464144,0,0,10617
9,2,washing_machine,321710,0,1464144,0,0,890


In [15]:
# ============================================================
# CELL 12 — CREATE ALIGNED MAINS–APPLIANCE PAIRS
# ============================================================

paired_data = {}

alignment_rows = []

print("Creating aligned mains–appliance pairs...")
print("=" * 100)

for house_id in HOUSE_IDS:

    print(f"\nHouse {house_id}")
    print("-" * 100)

    mains = cleaned_data[house_id]["mains"]

    paired_data[house_id] = {}

    for appliance in APPLIANCE_METERS[house_id]:

        appliance_series = cleaned_data[house_id][appliance]

        # ----------------------------------------------------
        # Find timestamps common to both signals
        # ----------------------------------------------------
        common_index = mains.index.intersection(
            appliance_series.index
        )

        mains_common = mains.loc[common_index]
        appliance_common = appliance_series.loc[common_index]

        # ----------------------------------------------------
        # Keep only timestamps where BOTH signals are valid
        # ----------------------------------------------------
        valid_mask = (
            mains_common.notna()
            & appliance_common.notna()
        )

        mains_valid = mains_common.loc[valid_mask]
        appliance_valid = appliance_common.loc[valid_mask]

        # ----------------------------------------------------
        # Safety check: indices must be identical
        # ----------------------------------------------------
        assert mains_valid.index.equals(
            appliance_valid.index
        )

        # ----------------------------------------------------
        # Store as a DataFrame
        # ----------------------------------------------------
        pair_df = pd.DataFrame({
            "mains": mains_valid,
            "appliance": appliance_valid
        })

        paired_data[house_id][appliance] = pair_df

        # ----------------------------------------------------
        # Record alignment statistics
        # ----------------------------------------------------
        original_common_samples = len(common_index)
        valid_samples = len(pair_df)
        removed_samples = (
            original_common_samples - valid_samples
        )

        alignment_rows.append({
            "house": house_id,
            "appliance": appliance,
            "common_samples_before_NaN_removal":
                original_common_samples,
            "valid_aligned_samples":
                valid_samples,
            "removed_due_to_NaN":
                removed_samples,
            "retained_percentage":
                valid_samples / original_common_samples * 100
        })

        print(
            f"{appliance:<20} | "
            f"before: {original_common_samples:>10,} | "
            f"valid: {valid_samples:>10,} | "
            f"removed: {removed_samples:>10,} | "
            f"retained: "
            f"{valid_samples / original_common_samples * 100:>6.2f}%"
        )


alignment_df = pd.DataFrame(alignment_rows)

print("\n")
print("ALIGNMENT SUMMARY")
print("=" * 100)

display(alignment_df)

Creating aligned mains–appliance pairs...

House 1
----------------------------------------------------------------------------------------------------
dishwasher           | before: 21,613,429 | valid: 21,218,580 | removed:    394,849 | retained:  98.17%
television           | before: 21,613,430 | valid: 21,216,359 | removed:    397,071 | retained:  98.16%
fridge_freezer       | before: 21,613,402 | valid: 21,214,063 | removed:    399,339 | retained:  98.15%
microwave            | before: 21,613,404 | valid: 21,214,848 | removed:    398,556 | retained:  98.16%
oven                 | before: 21,613,421 | valid: 20,874,753 | removed:    738,668 | retained:  96.58%

House 2
----------------------------------------------------------------------------------------------------
kettle               | before:  2,539,508 | valid:  1,931,950 | removed:    607,558 | retained:  76.08%
rice_cooker          | before:  2,539,179 | valid:  1,931,035 | removed:    608,144 | retained:  76.05%
washing_ma

,house,appliance,common_samples_before_NaN_removal,valid_aligned_samples,removed_due_to_NaN,retained_percentage
0,1,dishwasher,21613429,21218580,394849,98.173131
1,1,television,21613430,21216359,397071,98.162851
2,1,fridge_freezer,21613402,21214063,399339,98.152355
3,1,microwave,21613404,21214848,398556,98.155978
4,1,oven,21613421,20874753,738668,96.582364
5,2,kettle,2539508,1931950,607558,76.075760
6,2,rice_cooker,2539179,1931035,608144,76.049581
7,2,washing_machine,2049468,1513870,535598,73.866486
8,2,dishwasher,2049468,1513926,535542,73.869219
9,2,fridge,2049469,1513927,535542,73.869231


In [16]:
# ============================================================
# CELL 13 — VERIFY ALIGNED DATA AND CONTINUOUS SEGMENTS
# ============================================================

SEGMENT_GAP_SECONDS = SAMPLE_PERIOD


def analyze_continuous_segments(pair_df):
    """
    Analyze continuous sections of a paired mains-appliance
    dataset.
    """

    if len(pair_df) == 0:
        return {
            "segments": 0,
            "irregular_intervals": 0,
            "longest_segment_samples": 0,
            "shortest_segment_samples": 0
        }

    intervals = (
        pair_df.index.to_series()
        .diff()
        .dropna()
    )

    breaks = intervals != pd.Timedelta(
        seconds=SAMPLE_PERIOD
    )

    number_of_breaks = breaks.sum()

    segment_ids = breaks.cumsum()

    segment_lengths = (
        pair_df.groupby(segment_ids)
        .size()
    )

    return {
        "segments": len(segment_lengths),
        "irregular_intervals": int(number_of_breaks),
        "longest_segment_samples":
            int(segment_lengths.max()),
        "shortest_segment_samples":
            int(segment_lengths.min())
    }


segment_rows = []

print("ALIGNMENT AND CONTINUOUS-SEGMENT VERIFICATION")
print("=" * 110)

for house_id in HOUSE_IDS:

    print(f"\nHouse {house_id}")
    print("-" * 110)

    for appliance, pair_df in paired_data[house_id].items():

        # Basic checks
        assert pair_df.index.is_monotonic_increasing
        assert pair_df.index.is_unique
        assert pair_df["mains"].notna().all()
        assert pair_df["appliance"].notna().all()

        # Segment analysis
        segment_stats = analyze_continuous_segments(
            pair_df
        )

        row = {
            "house": house_id,
            "appliance": appliance,
            "samples": len(pair_df),
            "segments": segment_stats["segments"],
            "breaks": segment_stats["irregular_intervals"],
            "longest_segment_samples":
                segment_stats["longest_segment_samples"],
            "shortest_segment_samples":
                segment_stats["shortest_segment_samples"],
            "longest_segment_hours":
                (
                    segment_stats["longest_segment_samples"]
                    * SAMPLE_PERIOD
                    / 3600
                )
        }

        segment_rows.append(row)

        print(
            f"{appliance:<20} | "
            f"samples: {len(pair_df):>10,} | "
            f"segments: {segment_stats['segments']:>5,} | "
            f"breaks: {segment_stats['irregular_intervals']:>5,} | "
            f"longest: "
            f"{row['longest_segment_hours']:>8.2f} h"
        )


segment_df = pd.DataFrame(segment_rows)

print("\n")
print("SEGMENT SUMMARY")
print("=" * 110)

display(segment_df)

ALIGNMENT AND CONTINUOUS-SEGMENT VERIFICATION

House 1
--------------------------------------------------------------------------------------------------------------


dishwasher           | samples: 21,218,580 | segments:    43 | breaks:    42 | longest:  4829.97 h
television           | samples: 21,216,359 | segments:    23 | breaks:    22 | longest:  6965.38 h
fridge_freezer       | samples: 21,214,063 | segments:    94 | breaks:    93 | longest:  3273.45 h
microwave            | samples: 21,214,848 | segments:    84 | breaks:    83 | longest:  6809.99 h
oven                 | samples: 20,874,753 | segments:    69 | breaks:    68 | longest:  3677.47 h

House 2
--------------------------------------------------------------------------------------------------------------
kettle               | samples:  1,931,950 | segments:     8 | breaks:     7 | longest:  1819.73 h
rice_cooker          | samples:  1,931,035 | segments:    17 | breaks:    16 | longest:  1819.73 h
washing_machine      | samples:  1,513,870 | segments:     6 | breaks:     5 | longest:  1610.90 h
dishwasher           | samples:  1,513,926 | segments:     5 | breaks:     4 | longest: 

,house,appliance,samples,segments,breaks,longest_segment_samples,shortest_segment_samples,longest_segment_hours
0,1,dishwasher,21218580,43,42,2897979,21,4829.965000
1,1,television,21216359,23,22,4179228,229,6965.380000
2,1,fridge_freezer,21214063,94,93,1964072,21,3273.453333
3,1,microwave,21214848,84,83,4085996,21,6809.993333
4,1,oven,20874753,69,68,2206479,21,3677.465000
5,2,kettle,1931950,8,7,1091840,90,1819.733333
6,2,rice_cooker,1931035,17,16,1091840,21,1819.733333
7,2,washing_machine,1513870,6,5,966541,343,1610.901667
8,2,dishwasher,1513926,5,4,1091840,344,1819.733333
9,2,fridge,1513927,5,4,1091840,344,1819.733333


In [17]:
# ============================================================
# CELL 14 — FINAL FUNDAMENTAL CLEANING VALIDATION
# ============================================================

final_validation_rows = []

for house_id in HOUSE_IDS:
    for appliance, pair_df in paired_data[house_id].items():

        # ----------------------------------------------------
        # Basic integrity checks
        # ----------------------------------------------------
        assert pair_df.index.is_monotonic_increasing
        assert pair_df.index.is_unique

        assert pair_df["mains"].notna().all()
        assert pair_df["appliance"].notna().all()

        # ----------------------------------------------------
        # Identify continuous segments
        # ----------------------------------------------------
        intervals = (
            pair_df.index.to_series()
            .diff()
        )

        segment_breaks = (
            intervals != pd.Timedelta(seconds=SAMPLE_PERIOD)
        )

        # First sample is not an interval
        segment_breaks.iloc[0] = False

        segment_ids = segment_breaks.cumsum()

        # ----------------------------------------------------
        # Check intervals INSIDE each continuous segment
        # ----------------------------------------------------
        irregular_intervals = 0
        total_internal_intervals = 0

        for _, segment in pair_df.groupby(segment_ids):

            if len(segment) <= 1:
                continue

            segment_intervals = (
                segment.index.to_series()
                .diff()
                .dropna()
            )

            total_internal_intervals += len(
                segment_intervals
            )

            irregular_intervals += (
                segment_intervals
                != pd.Timedelta(seconds=SAMPLE_PERIOD)
            ).sum()

        exact_6sec_intervals = (
            total_internal_intervals
            - irregular_intervals
        )

        # ----------------------------------------------------
        # Signal sanity checks
        # ----------------------------------------------------
        mains_negative = (
            pair_df["mains"] < 0
        ).sum()

        appliance_negative = (
            pair_df["appliance"] < 0
        ).sum()

        mains_inf = np.isinf(
            pair_df["mains"].to_numpy()
        ).sum()

        appliance_inf = np.isinf(
            pair_df["appliance"].to_numpy()
        ).sum()

        # ----------------------------------------------------
        # Number of continuous segments
        # ----------------------------------------------------
        number_of_segments = (
            segment_ids.nunique()
        )

        # ----------------------------------------------------
        # Retention
        # ----------------------------------------------------
        original_samples = len(
            raw_data[house_id]["mains"]
        )

        retained_samples = len(pair_df)

        retention_percentage = (
            retained_samples
            / original_samples
        ) * 100

        final_validation_rows.append({
            "house": house_id,
            "appliance": appliance,
            "retained_samples": retained_samples,
            "retention_%": retention_percentage,
            "segments": number_of_segments,
            "exact_6sec_intervals":
                int(exact_6sec_intervals),
            "irregular_intervals":
                int(irregular_intervals),
            "mains_negative":
                int(mains_negative),
            "appliance_negative":
                int(appliance_negative),
            "mains_inf":
                int(mains_inf),
            "appliance_inf":
                int(appliance_inf)
        })

final_validation_df = pd.DataFrame(
    final_validation_rows
)

print("FINAL FUNDAMENTAL CLEANING VALIDATION")
print("=" * 120)

display(final_validation_df)

print("\nValidation checks:")
print("-" * 70)

print(
    "Maximum irregular intervals within segments:",
    final_validation_df["irregular_intervals"].max()
)

print(
    "Maximum negative mains values:",
    final_validation_df["mains_negative"].max()
)

print(
    "Maximum negative appliance values:",
    final_validation_df["appliance_negative"].max()
)

print(
    "Maximum infinite mains values:",
    final_validation_df["mains_inf"].max()
)

print(
    "Maximum infinite appliance values:",
    final_validation_df["appliance_inf"].max()
)

print("\nFundamental data cleaning validation completed.")

FINAL FUNDAMENTAL CLEANING VALIDATION


,house,appliance,retained_samples,retention_%,segments,exact_6sec_intervals,irregular_intervals,mains_negative,appliance_negative,mains_inf,appliance_inf
0,1,dishwasher,21218580,98.173113,43,21218537,0,0,0,0,0
1,1,television,21216359,98.162837,23,21216336,0,0,0,0,0
2,1,fridge_freezer,21214063,98.152214,94,21213969,0,0,0,0,0
3,1,microwave,21214848,98.155846,84,21214764,0,0,0,0,0
4,1,oven,20874753,96.582311,69,20874684,0,0,0,0,0
5,2,kettle,1931950,76.075730,8,1931942,0,0,0,0,0
6,2,rice_cooker,1931035,76.039699,17,1931018,0,0,0,0,0
7,2,washing_machine,1513870,59.612705,6,1513864,0,0,0,0,0
8,2,dishwasher,1513926,59.614910,5,1513921,0,0,0,0,0
9,2,fridge,1513927,59.614949,5,1513922,0,0,0,0,0



Validation checks:
----------------------------------------------------------------------
Maximum irregular intervals within segments: 0
Maximum negative mains values: 0
Maximum negative appliance values: 0
Maximum infinite mains values: 0
Maximum infinite appliance values: 0

Fundamental data cleaning validation completed.


In [18]:
import os
import pandas as pd

# Create processed-data directory if it does not already exist
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(
    PROCESSED_DIR,
    "ukdale_fundamental_cleaned_6s.h5"
)

# Save each house/appliance pair as a separate HDF5 table
with pd.HDFStore(OUTPUT_PATH, mode="w") as store:
    
    for house_id in paired_data:
        for appliance, pair_df in paired_data[house_id].items():
            
            key = f"house_{house_id}/{appliance}"
            
            store.put(
                key,
                pair_df,
                format="table",
                data_columns=True
            )

print("Cleaned dataset saved successfully.")
print("File:", OUTPUT_PATH)

# Display saved contents
with pd.HDFStore(OUTPUT_PATH, mode="r") as store:
    print("\nSaved datasets:")
    for key in store.keys():
        print(" ", key)

Cleaned dataset saved successfully.
File: ../data/processed\ukdale_fundamental_cleaned_6s.h5

Saved datasets:
  /house_5/electric_oven
  /house_5/fridge_freezer
  /house_5/washer_dryer
  /house_2/dishwasher
  /house_2/fridge
  /house_2/kettle
  /house_2/rice_cooker
  /house_2/washing_machine
  /house_1/dishwasher
  /house_1/fridge_freezer
  /house_1/microwave
  /house_1/oven
  /house_1/television


In [19]:
import os
import pandas as pd

OUTPUT_PATH = "../data/processed/ukdale_fundamental_cleaned_6s.h5"

print("File exists:", os.path.exists(OUTPUT_PATH))

with pd.HDFStore(OUTPUT_PATH, mode="r") as store:
    print("\nSaved dataset structure:")
    for key in store.keys():
        df = store[key]
        print(
            f"{key}: "
            f"{len(df):,} rows | "
            f"columns = {list(df.columns)}"
        )

File exists: True

Saved dataset structure:
/house_5/electric_oven: 1,900,727 rows | columns = ['mains', 'appliance']
/house_5/fridge_freezer: 1,900,820 rows | columns = ['mains', 'appliance']
/house_5/washer_dryer: 1,900,814 rows | columns = ['mains', 'appliance']
/house_2/dishwasher: 1,513,926 rows | columns = ['mains', 'appliance']
/house_2/fridge: 1,513,927 rows | columns = ['mains', 'appliance']
/house_2/kettle: 1,931,950 rows | columns = ['mains', 'appliance']
/house_2/rice_cooker: 1,931,035 rows | columns = ['mains', 'appliance']
/house_2/washing_machine: 1,513,870 rows | columns = ['mains', 'appliance']
/house_1/dishwasher: 21,218,580 rows | columns = ['mains', 'appliance']
/house_1/fridge_freezer: 21,214,063 rows | columns = ['mains', 'appliance']
/house_1/microwave: 21,214,848 rows | columns = ['mains', 'appliance']
/house_1/oven: 20,874,753 rows | columns = ['mains', 'appliance']
/house_1/television: 21,216,359 rows | columns = ['mains', 'appliance']
